# LayerNorm Ablation: Impact on Riemannian Interpretability

## Goal

The Arm 2 diagnostic revealed that damped geodesic **directional** compliance is
strong (cosine ≈ 0.73–0.75) but **magnitude** compliance is catastrophically poor
($R^2 < 0$). The primary culprit is LayerNorm, which resets the kinetic energy at
every step and introduces a non-conservative projection force.

This notebook trains four identical SPLM checkpoints with different normalisation
strategies and re-runs the Arm 2 diagnostic to quantify the impact:

| Option | `_project` replacement | Expected geometric impact |
|--------|------------------------|---------------------------|
| **A — RMSNorm** | $h / \text{RMS}(h)$ | Moderate; conformal factor computable |
| **B — Energy-shell** | $h \cdot \sqrt{(E_0 e^{-\gamma \ell} - V) / T}$ | Large; exact damped geodesics |
| **C — Spectral norm** | Identity (normalise weights) | Full preservation |
| **D — No norm** | Identity + grad clip | Best geometry; highest training risk |

A baseline run with the original LayerNorm is included for comparison.

## Reference

- Companion note §3.7: *Impact of LayerNorm on Riemannian Interpretability*
- Diagnostic battery: `colab_riemannian_diagnostic.ipynb` (Arm 2)

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import asdict, fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow scipy scikit-learn')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.parametrize as parametrize
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')

# ── GDrive / local results directory ──
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_layernorm_ablation')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_layernorm_ablation'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_CKPTS = DRIVE_ROOT / 'checkpoints'
DRIVE_CKPTS.mkdir(parents=True, exist_ok=True)
print(f'Drive root   : {DRIVE_ROOT}')
print(f'Results dir  : {DRIVE_RESULTS}')
print(f'Checkpoints  : {DRIVE_CKPTS}')

In [ ]:
# ── Cell 2: Load data ─────────────────────────────────────────────
from data_module import get_batch, load_tiny_stories

SCRIPTS_DIR = os.path.join(ARCH_DIR, 'scaleup')
LOGFREQ_PATH = os.path.join(SCRIPTS_DIR, 'results', 'logfreq_surprisal_tinystories.npy')

if not os.path.exists(LOGFREQ_PATH):
    print('Computing logfreq surprisal (one-time, ~2 min)...')
    os.makedirs(os.path.dirname(LOGFREQ_PATH), exist_ok=True)
    subprocess.run(
        [sys.executable, os.path.join(SCRIPTS_DIR, 'compute_unigram_frequencies_tinystories.py')],
        cwd=SCRIPTS_DIR, check=True,
    )
    print('Done.')

print('Loading TinyStories...')
train_ids, val_ids = load_tiny_stories()
print(f'Train tokens: {len(train_ids):,}  Val tokens: {len(val_ids):,}')

rng = np.random.default_rng(42)

In [ ]:
# ── Cell 3: Model imports + normalisation strategies ──────────────
from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)


# ── Normalisation replacement functions ──
# Each returns a closure that replaces model._project

def make_layernorm_project(model):
    """Baseline: original LayerNorm (no change)."""
    d = model.cfg.d
    eps = model.cfg.ln_eps
    def _project(h):
        return F.layer_norm(h, (d,), eps=eps)
    return _project


def make_rmsnorm_project(model):
    """Option A: RMSNorm — rescale by root-mean-square, no mean subtraction."""
    d = model.cfg.d
    eps = model.cfg.ln_eps
    def _project(h):
        rms = torch.sqrt(torch.mean(h * h, dim=-1, keepdim=True) + eps)
        return h / rms
    return _project


def make_energy_shell_project(model):
    """Option B: Energy-shell projection.

    After each step, rescale h so that E = T + V stays on the
    prescribed damping curve E_0 * exp(-gamma * ell).

    NOTE: This requires the layer index at call time. We attach a mutable
    counter to the model and increment it inside _project. The integrate()
    loop calls _project once per layer, so the counter tracks the layer.
    The counter is reset at the start of each forward pass via a hook.
    """
    model._eshell_layer_counter = 0
    model._eshell_E0 = None  # set after first layer
    gamma = model.gamma.item() if hasattr(model.gamma, 'item') else float(model.gamma)

    def _project(h):
        ell = model._eshell_layer_counter
        model._eshell_layer_counter = ell + 1

        # We need V_theta at h. Use detached h for the xi channels to
        # avoid breaking the autograd graph for the main integration.
        with torch.no_grad():
            h_det = h.detach()
            xis = model.xi_module(h_det)
            V = model.V_theta(xis, h_det)  # (B, T, 1)

        # Kinetic energy of current step
        # We don't have v directly, so estimate from h norm change.
        # For the first call (ell=0) we just return h unchanged (initial projection).
        if ell == 0:
            # Record initial energy for the damping curve
            T_init = 0.5 * (h * h).sum(dim=-1, keepdim=True).mean()
            model._eshell_E0 = (T_init + V.mean()).item()
            return h

        E0 = model._eshell_E0
        if E0 is None or E0 <= 0:
            return h

        E_target = E0 * math.exp(-gamma * ell)
        T_current = 0.5 * (h * h).sum(dim=-1, keepdim=True)
        E_current = T_current + V

        # Rescale h so that T_new + V ≈ E_target
        # T_new = E_target - V => scale² = (E_target - V) / T_current
        T_target = E_target - V
        # Clamp to avoid negative kinetic energy
        ratio = (T_target / T_current.clamp(min=1e-8)).clamp(min=0.01, max=100.0)
        scale = torch.sqrt(ratio)
        return h * scale

    return _project


def make_identity_project(model):
    """Options C & D: no activation normalisation (identity)."""
    def _project(h):
        return h
    return _project


def apply_spectral_norm_to_vtheta(model):
    """Option C: Apply spectral normalisation to all Linear layers in V_theta."""
    for name, module in model.V_theta.named_modules():
        if isinstance(module, nn.Linear):
            parametrize.register_parametrization(module, 'weight',
                torch.nn.utils.parametrizations.spectral_norm(module).parametrizations.weight[0])
            print(f'  Spectral norm applied to V_theta.{name}')


def apply_spectral_norm_simple(model):
    """Option C: Apply spectral normalisation to all Linear layers in V_theta.
    Uses the classic torch.nn.utils.spectral_norm API for compatibility."""
    count = 0
    for name, module in list(model.V_theta.named_modules()):
        if isinstance(module, nn.Linear):
            # Navigate to parent to replace the submodule
            parts = name.split('.')
            parent = model.V_theta
            for p in parts[:-1]:
                parent = getattr(parent, p)
            setattr(parent, parts[-1], nn.utils.spectral_norm(module))
            count += 1
    print(f'  Spectral norm applied to {count} Linear layers in V_theta')


# ── Ablation configurations ──
ABLATION_OPTIONS = {
    'baseline_layernorm': {
        'label': 'Baseline (LayerNorm)',
        'project_fn': make_layernorm_project,
        'weight_norm': False,
        'grad_clip': 1.0,
        'color': 'tab:blue',
    },
    'rmsnorm': {
        'label': 'A: RMSNorm',
        'project_fn': make_rmsnorm_project,
        'weight_norm': False,
        'grad_clip': 1.0,
        'color': 'tab:orange',
    },
    'energy_shell': {
        'label': 'B: Energy-Shell',
        'project_fn': make_energy_shell_project,
        'weight_norm': False,
        'grad_clip': 1.0,
        'color': 'tab:green',
    },
    'spectral_norm': {
        'label': 'C: Spectral Norm',
        'project_fn': make_identity_project,
        'weight_norm': True,
        'grad_clip': 1.0,
        'color': 'tab:red',
    },
    'no_norm': {
        'label': 'D: No Norm + Clip',
        'project_fn': make_identity_project,
        'weight_norm': False,
        'grad_clip': 1.0,
        'color': 'tab:purple',
    },
}

print(f'Defined {len(ABLATION_OPTIONS)} ablation options:')
for k, v in ABLATION_OPTIONS.items():
    print(f'  {k}: {v["label"]}')

In [ ]:
# ── Cell 4: Training configuration ────────────────────────────────
# Reduced-scale config: d=128, L=6, 2000 steps.
# ~1.5M params per model, trains in ~15-20 min on T4.
# Total wall time for all 5 options: ~1.5 hours.

TRAIN_STEPS = 2000
EVAL_INTERVAL = 200
EVAL_ITERS = 20
LOG_INTERVAL = 50
BATCH_SIZE = 8
BLOCK_SIZE = 256
LR = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100

MODEL_D = 128
MODEL_L = 6
V_HIDDEN = 512
V_DEPTH = 3
XI_CHANNELS = 4

print(f'Training config: {TRAIN_STEPS} steps, bs={BATCH_SIZE}, T={BLOCK_SIZE}')
print(f'Model config: d={MODEL_D}, L={MODEL_L}, v_hidden={V_HIDDEN}, v_depth={V_DEPTH}')
print(f'Estimated wall time per option: ~15-20 min on T4, ~8-10 min on A100')
print(f'Total for all {len(ABLATION_OPTIONS)} options: ~1.5-2 hours on T4')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

def lr_schedule(step, lr, warmup, total):
    if step < warmup:
        return lr * (step + 1) / warmup
    progress = (step - warmup) / max(total - warmup, 1)
    return lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


@torch.no_grad()
def evaluate(model, ids, iters, batch_size, block_size, rng_eval, device):
    model.eval()
    losses = []
    for _ in range(iters):
        xb, yb = get_batch(ids, batch_size, block_size, rng_eval)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def free_mem(model=None):
    if model is not None:
        model.cpu()
        del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


def train_one_option(option_key, option_cfg, seed=42):
    """Train a single SPLM model with the given normalisation option."""
    label = option_cfg['label']
    ckpt_path = DRIVE_CKPTS / f'{option_key}.pt'

    # Skip if checkpoint already exists
    if ckpt_path.exists():
        print(f'\n  [{label}] Checkpoint exists at {ckpt_path}, skipping training.')
        return ckpt_path

    print(f'\n{"═" * 60}')
    print(f'Training: {label}')
    print(f'{"═" * 60}')

    torch.manual_seed(seed)
    np.random.seed(seed)

    model_cfg = SPLMSARFMassLNMultiXiConfig(
        d=MODEL_D, max_len=1024,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH, L=MODEL_L,
        init_m=1.0, init_gamma=1.0,
        vocab_size=50257,
        mass_mode='logfreq',
        logfreq_init_alpha=0.1,
        logfreq_path=LOGFREQ_PATH,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=[0.0, 0.5, 0.9, 0.99],
        xi_learnable=True,
    )
    model = ScalarPotentialLMSARFMassLNMultiXi(model_cfg)

    # Apply weight normalisation if needed (Option C)
    if option_cfg['weight_norm']:
        apply_spectral_norm_simple(model)

    # Replace _project with the chosen normalisation
    import types
    new_project = option_cfg['project_fn'](model)
    model._project = types.MethodType(lambda self, h, _fn=new_project: _fn(h), model)

    # For energy-shell: reset counter at each forward pass
    if option_key == 'energy_shell':
        _orig_forward = model.forward
        def _patched_forward(*args, **kwargs):
            model._eshell_layer_counter = 0
            model._eshell_E0 = None
            return _orig_forward(*args, **kwargs)
        model.forward = _patched_forward

    model.to(DEVICE).train()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Parameters: {n_params:.2f}M')

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
    )
    grad_clip = option_cfg['grad_clip']
    rng_train = np.random.default_rng(seed)
    rng_eval = np.random.default_rng(seed + 1)

    train_losses = []
    val_ppls = []
    t0 = time.time()

    for step in range(TRAIN_STEPS):
        lr_now = lr_schedule(step, LR, WARMUP_STEPS, TRAIN_STEPS)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now

        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng_train)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)

        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        train_losses.append(loss.item())

        if step % LOG_INTERVAL == 0:
            elapsed = time.time() - t0
            print(f'  step {step:5d}/{TRAIN_STEPS}  loss={loss.item():.4f}  '
                  f'lr={lr_now:.2e}  elapsed={elapsed:.0f}s')

        if (step + 1) % EVAL_INTERVAL == 0 or step == TRAIN_STEPS - 1:
            val_loss = evaluate(model, val_ids, EVAL_ITERS, BATCH_SIZE,
                                BLOCK_SIZE, rng_eval, DEVICE)
            val_ppl = math.exp(min(val_loss, 20.0))
            val_ppls.append((step, val_ppl))
            print(f'  [eval] step {step+1}  val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}')

        del x, y, loss

    elapsed = time.time() - t0
    print(f'  Training complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')

    # Save checkpoint
    torch.save({
        'config': asdict(model_cfg),
        'model_state_dict': model.state_dict(),
        'option_key': option_key,
        'option_label': label,
        'train_losses': train_losses,
        'val_ppls': val_ppls,
        'train_steps': TRAIN_STEPS,
        'elapsed_s': elapsed,
    }, ckpt_path)
    print(f'  Saved checkpoint: {ckpt_path}')

    free_mem(model)
    return ckpt_path


# ── Train all options ──
ckpt_paths = {}
for key, cfg in ABLATION_OPTIONS.items():
    ckpt_paths[key] = train_one_option(key, cfg)

print(f'\n{"═" * 60}')
print('All training runs complete.')
for k, p in ckpt_paths.items():
    print(f'  {k}: {p}')

In [ ]:
# ── Cell 6: Training curves comparison ────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for key, cfg in ABLATION_OPTIONS.items():
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    label = cfg['label']
    color = cfg['color']

    # Smoothed training loss
    tl = np.array(ckpt['train_losses'])
    window = min(50, len(tl) // 4)
    if window > 1:
        tl_smooth = np.convolve(tl, np.ones(window)/window, mode='valid')
        axes[0].plot(range(window-1, len(tl)), tl_smooth, label=label, color=color, alpha=0.8)
    else:
        axes[0].plot(tl, label=label, color=color, alpha=0.8)

    # Val PPL
    vp = ckpt['val_ppls']
    steps_v = [x[0] for x in vp]
    ppls_v = [x[1] for x in vp]
    axes[1].plot(steps_v, ppls_v, 'o-', label=label, color=color, markersize=4)

axes[0].set_xlabel('Training Step'); axes[0].set_ylabel('Loss (smoothed)')
axes[0].set_title('Training Loss'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('Training Step'); axes[1].set_ylabel('Val PPL')
axes[1].set_title('Validation Perplexity'); axes[1].legend(fontsize=8)

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('LayerNorm Ablation: Training Dynamics', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'training_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "training_curves.png"}')

In [ ]:
# ── Cell 7: Arm 2 Diagnostic — Geodesic Compliance ────────────────
# Re-run Arm 2 (cosine + R²-magnitude compliance) for each checkpoint.
# Uses the SAME code as the original diagnostic battery.

import pyarrow.parquet as pq
from data_module import _download_hf_parquet, _resolve_tinystories_shard, _gpt2_tokenize

# Prepare small eval batches
N_EVAL_BATCHES = 3
EVAL_BATCH_SIZE = 4
EVAL_BLOCK_SIZE = 128

rng_diag = np.random.default_rng(123)
eval_batches = []
for _ in range(N_EVAL_BATCHES):
    xb, _ = get_batch(val_ids, EVAL_BATCH_SIZE, EVAL_BLOCK_SIZE, rng_diag)
    eval_batches.append(torch.tensor(xb))


def load_ablation_model(option_key, option_cfg):
    """Load a trained ablation checkpoint and re-apply the normalisation patch."""
    import types
    ckpt = torch.load(ckpt_paths[option_key], map_location='cpu', weights_only=False)
    cfg_dict = ckpt['config']
    known = {f.name for f in dc_fields(SPLMSARFMassLNMultiXiConfig)}
    model_cfg = SPLMSARFMassLNMultiXiConfig(
        **{k: v for k, v in cfg_dict.items() if k in known}
    )
    if hasattr(model_cfg, 'logfreq_path'):
        model_cfg.logfreq_path = LOGFREQ_PATH

    model = ScalarPotentialLMSARFMassLNMultiXi(model_cfg)

    # Apply weight normalisation BEFORE loading state dict if needed
    if option_cfg['weight_norm']:
        apply_spectral_norm_simple(model)

    model.load_state_dict(ckpt['model_state_dict'], strict=False)

    # Replace _project
    new_project = option_cfg['project_fn'](model)
    model._project = types.MethodType(lambda self, h, _fn=new_project: _fn(h), model)

    # Energy-shell forward patch
    if option_key == 'energy_shell':
        _orig_forward = model.forward
        def _patched_forward(*args, **kwargs):
            model._eshell_layer_counter = 0
            model._eshell_E0 = None
            return _orig_forward(*args, **kwargs)
        model.forward = _patched_forward

    model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Loaded {option_cfg["label"]}: {n:.2f}M params')
    return model


def extract_trajectory(model, x):
    with torch.enable_grad():
        out = model(x, targets=None, return_trajectory=True,
                    return_xi_trajectory=False)
        logits, _loss, traj = out[0], out[1], out[2]
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    traj_cpu = [h.detach().cpu() for h in traj]
    logits_cpu = logits.detach().cpu()
    del traj, out, logits
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return traj_cpu, logits_cpu


def compute_grad_V(model, h):
    h_in = h.detach().requires_grad_(True)
    xis = model.xi_module(h_in.detach())
    V = model.V_theta(xis, h_in)
    grad_V = torch.autograd.grad(V.sum(), h_in, create_graph=False)[0]
    return grad_V.detach()


def get_mass(model, x):
    emb = model._embed(x)
    m = model.compute_mass(x, emb)
    if isinstance(m, torch.Tensor):
        return m.detach().cpu()
    return m


print('═' * 60)
print('ARM 2 DIAGNOSTIC: Geodesic Compliance (all ablation options)')
print('═' * 60)

arm2_all = {}

for option_key, option_cfg in ABLATION_OPTIONS.items():
    label = option_cfg['label']
    print(f'\n── {label} ──')

    model = load_ablation_model(option_key, option_cfg)

    gamma = model.gamma.item() if hasattr(model.gamma, 'item') else float(model.gamma)
    dt = getattr(getattr(model, 'cfg', None), 'dt', 1.0)
    print(f'  γ = {gamma:.6f}, dt = {dt}')

    comp_undamped_all = None
    comp_damped_all = None
    cos_undamped_all = None
    cos_damped_all = None

    for bi in range(N_EVAL_BATCHES):
        x = eval_batches[bi].to(DEVICE)
        traj, _ = extract_trajectory(model, x)
        L = len(traj) - 1

        if comp_undamped_all is None:
            comp_undamped_all = [[] for _ in range(L - 1)]
            comp_damped_all   = [[] for _ in range(L - 1)]
            cos_undamped_all  = [[] for _ in range(L - 1)]
            cos_damped_all    = [[] for _ in range(L - 1)]

        with torch.no_grad():
            m = get_mass(model, x)
            m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
        del x

        for ell in range(1, L):
            h_prev = traj[ell - 1].to(DEVICE)
            h_curr = traj[ell].to(DEVICE)
            h_next = traj[ell + 1].to(DEVICE)

            v = h_curr - h_prev
            a_obs = h_next - 2 * h_curr + h_prev
            del h_prev, h_next

            grad_V = compute_grad_V(model, h_curr)

            v_norm2 = (v ** 2).sum(dim=-1, keepdim=True)
            mf = (m_flat.unsqueeze(-1).to(DEVICE)
                  if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2
                  else m_flat)
            ke = 0.5 * mf * v_norm2
            denom = (2.0 * ke).clamp(min=1e-8)

            gV_dot_v = (grad_V * v).sum(dim=-1, keepdim=True)
            a_jacobi_undamped = (2.0 * v * gV_dot_v - v_norm2 * grad_V) / denom
            a_jacobi_damped = a_jacobi_undamped - gamma * v

            a2 = (a_obs ** 2).sum(dim=-1).mean().item()
            R2_und = ((a_obs - a_jacobi_undamped) ** 2).sum(dim=-1).mean().item()
            R2_dmp = ((a_obs - a_jacobi_damped) ** 2).sum(dim=-1).mean().item()
            comp_undamped_all[ell - 1].append(1.0 - R2_und / max(a2, 1e-12))
            comp_damped_all[ell - 1].append(1.0 - R2_dmp / max(a2, 1e-12))

            a_obs_flat = a_obs.reshape(-1, a_obs.shape[-1])
            und_flat = a_jacobi_undamped.reshape(-1, a_jacobi_undamped.shape[-1])
            dmp_flat = a_jacobi_damped.reshape(-1, a_jacobi_damped.shape[-1])
            cos_und = F.cosine_similarity(a_obs_flat, und_flat, dim=-1).mean().item()
            cos_dmp = F.cosine_similarity(a_obs_flat, dmp_flat, dim=-1).mean().item()
            cos_undamped_all[ell - 1].append(cos_und)
            cos_damped_all[ell - 1].append(cos_dmp)

            del h_curr, v, a_obs, grad_V, a_jacobi_undamped, a_jacobi_damped

        del traj
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    # Aggregate
    r2_und_mean = np.mean([np.mean(l) for l in comp_undamped_all])
    r2_dmp_mean = np.mean([np.mean(l) for l in comp_damped_all])
    cos_und_mean = np.mean([np.mean(l) for l in cos_undamped_all])
    cos_dmp_mean = np.mean([np.mean(l) for l in cos_damped_all])

    arm2_all[option_key] = {
        'label': label,
        'r2_undamped_per_layer': [np.mean(l) for l in comp_undamped_all],
        'r2_damped_per_layer':   [np.mean(l) for l in comp_damped_all],
        'cos_undamped_per_layer': [np.mean(l) for l in cos_undamped_all],
        'cos_damped_per_layer':   [np.mean(l) for l in cos_damped_all],
        'r2_undamped_mean': r2_und_mean,
        'r2_damped_mean': r2_dmp_mean,
        'cos_undamped_mean': cos_und_mean,
        'cos_damped_mean': cos_dmp_mean,
    }

    print(f'  R²-magnitude (undamped): {r2_und_mean:.4f}')
    print(f'  R²-magnitude (damped):   {r2_dmp_mean:.4f}')
    print(f'  Cosine (undamped):        {cos_und_mean:.4f}')
    print(f'  Cosine (damped):          {cos_dmp_mean:.4f}')

    free_mem(model); del model

print('\nArm 2 diagnostic complete for all options. ✓')

In [ ]:
# ── Cell 8: Arm 2 results — plots and summary ─────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for key, res in arm2_all.items():
    color = ABLATION_OPTIONS[key]['color']
    label = res['label']
    layers = range(1, len(res['r2_damped_per_layer']) + 1)

    axes[0, 0].plot(layers, res['cos_undamped_per_layer'], 'o--', label=label,
                    color=color, markersize=3, alpha=0.7)
    axes[0, 1].plot(layers, res['cos_damped_per_layer'], 'o-', label=label,
                    color=color, markersize=3)
    axes[1, 0].plot(layers, res['r2_undamped_per_layer'], 's--', label=label,
                    color=color, markersize=3, alpha=0.7)
    axes[1, 1].plot(layers, res['r2_damped_per_layer'], 's-', label=label,
                    color=color, markersize=3)

axes[0, 0].set_title('Cosine Compliance (Undamped)'); axes[0, 0].set_ylabel('Cosine Sim')
axes[0, 1].set_title('Cosine Compliance (Damped)'); axes[0, 1].set_ylabel('Cosine Sim')
axes[1, 0].set_title('R²-Magnitude Compliance (Undamped)'); axes[1, 0].set_ylabel('R²')
axes[1, 1].set_title('R²-Magnitude Compliance (Damped)'); axes[1, 1].set_ylabel('R²')

for ax in axes.flat:
    ax.set_xlabel('Layer')
    ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[1, 0].axhline(0, color='gray', ls=':', lw=0.8)
axes[1, 1].axhline(0, color='gray', ls=':', lw=0.8)

plt.suptitle('LayerNorm Ablation: Arm 2 Geodesic Compliance',
             fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'arm2_ablation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "arm2_ablation.png"}')


# ── Summary bar chart ──
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

keys = list(arm2_all.keys())
labels = [arm2_all[k]['label'] for k in keys]
colors = [ABLATION_OPTIONS[k]['color'] for k in keys]
cos_vals = [arm2_all[k]['cos_damped_mean'] for k in keys]
r2_vals  = [arm2_all[k]['r2_damped_mean'] for k in keys]

x_pos = range(len(keys))
axes2[0].bar(x_pos, cos_vals, color=colors)
axes2[0].set_xticks(x_pos); axes2[0].set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
axes2[0].set_ylabel('Mean Cosine Similarity')
axes2[0].set_title('Damped Cosine Compliance (Direction)')

axes2[1].bar(x_pos, r2_vals, color=colors)
axes2[1].set_xticks(x_pos); axes2[1].set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
axes2[1].set_ylabel('Mean R²')
axes2[1].set_title('Damped R²-Magnitude Compliance')
axes2[1].axhline(0, color='gray', ls=':', lw=0.8)

for ax in axes2:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('LayerNorm Ablation: Summary', fontweight='bold', y=1.02)
plt.tight_layout()
fig2.savefig(DRIVE_RESULTS / 'arm2_ablation_summary.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "arm2_ablation_summary.png"}')

In [ ]:
# ── Cell 9: Arm 1 quick check — Metric Validity ───────────────────
# Verify Ω² > 0 and check the conformal factor profile for each option.

print('═' * 60)
print('ARM 1 CHECK: Metric Validity (Ω² = 2T·m > 0)')
print('═' * 60)

arm1_all = {}

for option_key, option_cfg in ABLATION_OPTIONS.items():
    label = option_cfg['label']
    print(f'\n── {label} ──')

    model = load_ablation_model(option_key, option_cfg)
    x = eval_batches[0].to(DEVICE)
    traj, _ = extract_trajectory(model, x)
    L = len(traj) - 1

    with torch.no_grad():
        m = get_mass(model, x)
        m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
    del x

    frac_pos, omega_means = [], []
    for ell in range(1, L + 1):
        v = traj[ell] - traj[ell - 1]
        v_norm2 = (v ** 2).sum(dim=-1)
        mf = (m_flat if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2
              else (m_flat.expand(v_norm2.shape) if isinstance(m_flat, torch.Tensor)
                    else torch.full_like(v_norm2, float(m_flat))))
        omega2 = (mf ** 2) * v_norm2
        frac_pos.append((omega2 > 1e-12).float().mean().item())
        omega_means.append(omega2.mean().item())

    arm1_all[option_key] = {'frac_positive': frac_pos, 'omega2_mean': omega_means}
    print(f'  Ω²>0: {["{:.4f}".format(f) for f in frac_pos]}')
    print(f'  Mean Ω²: {["{:.2e}".format(v) for v in omega_means]}')

    del traj
    free_mem(model); del model

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, res in arm1_all.items():
    color = ABLATION_OPTIONS[key]['color']
    label = ABLATION_OPTIONS[key]['label']
    layers = range(1, len(res['frac_positive']) + 1)
    axes[0].plot(layers, res['frac_positive'], 'o-', label=label, color=color, markersize=3)
    axes[1].plot(layers, res['omega2_mean'],   'o-', label=label, color=color, markersize=3)

axes[0].set_ylabel('Fraction Ω² > 0'); axes[0].set_xlabel('Layer')
axes[0].set_title('Metric Validity'); axes[0].legend(fontsize=7)
axes[0].set_ylim(0.9, 1.01)
axes[1].set_ylabel('Mean Ω²'); axes[1].set_xlabel('Layer')
axes[1].set_title('Conformal Factor Profile'); axes[1].legend(fontsize=7)
for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('LayerNorm Ablation: Metric Validity Check', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'arm1_ablation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "arm1_ablation.png"}')
print('\nArm 1 check complete. ✓')

In [ ]:
# ── Cell 10: Save full results JSON ───────────────────────────────

# Collect final val PPL for each option
final_ppls = {}
for key in ABLATION_OPTIONS:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    vp = ckpt['val_ppls']
    final_ppls[key] = vp[-1][1] if vp else float('nan')

results = {
    'config': {
        'model_d': MODEL_D, 'model_L': MODEL_L,
        'v_hidden': V_HIDDEN, 'v_depth': V_DEPTH,
        'train_steps': TRAIN_STEPS, 'batch_size': BATCH_SIZE,
        'block_size': BLOCK_SIZE,
    },
    'options': {},
}

for key in ABLATION_OPTIONS:
    results['options'][key] = {
        'label': ABLATION_OPTIONS[key]['label'],
        'final_val_ppl': final_ppls[key],
        'arm2': arm2_all.get(key, {}),
        'arm1': arm1_all.get(key, {}),
    }

report_path = DRIVE_RESULTS / 'layernorm_ablation_report.json'
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Full results saved to: {report_path}')


# ── Summary table ──
print('\n' + '═' * 80)
print('LAYERNORM ABLATION: SUMMARY')
print('═' * 80)
print(f'{"Option":<25} {"Val PPL":>10} {"Cos(damp)":>12} {"R²(damp)":>12} {"Cos(undamp)":>14} {"R²(undamp)":>14}')
print('-' * 90)
for key in ABLATION_OPTIONS:
    a2 = arm2_all.get(key, {})
    print(f'{ABLATION_OPTIONS[key]["label"]:<25} '
          f'{final_ppls[key]:>10.2f} '
          f'{a2.get("cos_damped_mean", float("nan")):>12.4f} '
          f'{a2.get("r2_damped_mean", float("nan")):>12.4f} '
          f'{a2.get("cos_undamped_mean", float("nan")):>14.4f} '
          f'{a2.get("r2_undamped_mean", float("nan")):>14.4f}')
print('═' * 80)
print('\nKey questions:')
print('  1. Does any option bring R²-magnitude compliance > 0?')
print('  2. Does cosine compliance improve or hold steady?')
print('  3. What is the val PPL cost of each alternative?')
print('\n✓ Ablation complete.')